# 01 Data Loading and Graph Construction
Reads all raw CSVs and builds the full multilayer supra-graph `G_full`.

In [1]:
import sys
sys.path.insert(0, '..')   # so 'src' is importable from notebooks/

import pandas as pd

from src.network.build import build_isp_graph, attach_ixps, prepare_edge_df

In [2]:
# file paths
import yaml
with open("../config.yaml", "r") as file:
    config = yaml.safe_load(file)

# Access individual constant values like a dictionary
NODE_DIR = config["paths"]["node_list"]
EDGE_DIR = config["paths"]["edge_list"]
IXP_DIR = config["paths"]["ixp_list"]
SRG_DIR = config["paths"]["srg_list"]
SITE_DIR = config["paths"]["site_list"]
GRAPH_DIR = config["paths"]["graph_folder"]

In [3]:
# Load raw data
node_df = pd.read_csv(NODE_DIR)
edge_df = pd.read_csv(EDGE_DIR)
srg_city_df = pd.read_csv(SRG_DIR)
ixp_df = pd.read_csv(IXP_DIR)
site_df = pd.read_csv(SITE_DIR, comment='#')

# add composite edge key
edge_df = prepare_edge_df(edge_df)

# print stats
print('Nodes:', len(node_df), '| Edges:', len(edge_df),
      '| Sites:', len(site_df), '| IXPs:', len(ixp_df))

Nodes: 59 | Edges: 132 | Sites: 16 | IXPs: 8


In [4]:
# Build graph object
G_full = build_isp_graph(node_df, edge_df, site_df)
G_full = attach_ixps(G_full, ixp_df, site_df)

print(G_full)

MultiGraph with 67 nodes and 171 edges


In [5]:
# Sanity check: node count per ISP
from collections import Counter
Counter(data['isp'] for _, data in G_full.nodes(data=True))

Counter({'optus': 12,
         'aarnet': 11,
         'vocus': 11,
         'abb': 9,
         'superloop': 8,
         'telstra': 8,
         'ixp': 8})

In [6]:
# save the graph object
import pickle
import os
os.makedirs(GRAPH_DIR, exist_ok=True)
with open(f"{GRAPH_DIR}/G_full.pickle", "wb") as f:
    pickle.dump(G_full, f)